In [12]:
from langgraph.graph import StateGraph, START, END, add_messages
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, Annotated
from langgraph.checkpoint.memory import MemorySaver
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
model = ChatGoogleGenerativeAI(model = 'gemini-2.5-flash')

In [4]:
class ChatState(TypedDict):
    
    messages: Annotated[list[BaseMessage], add_messages]

In [5]:
def chatbot(state: ChatState) -> ChatState:
    response = model.invoke(state['messages'])
    
    return {'messages': [response.content]}

In [13]:
checkpointer = MemorySaver()
graph = StateGraph(ChatState)

graph.add_node('Chatbot', chatbot)

graph.add_edge(START, 'Chatbot')
graph.add_edge('Chatbot', END)

workflow = graph.compile(checkpointer= checkpointer)

In [7]:
initial_state = {
    'messages': [HumanMessage(content='what is langchain?')]
}

In [8]:
final_state = workflow.invoke(initial_state)

In [9]:
final_state

{'messages': [HumanMessage(content='what is langchain?', additional_kwargs={}, response_metadata={}, id='a275d739-422f-4cb3-85b2-3f070b543d95'),
  HumanMessage(content='**LangChain is a powerful open-source framework designed to help developers build applications powered by Large Language Models (LLMs).**\n\nThink of it as an **orchestration layer** or a **toolkit** that makes it easier to connect LLMs with external data sources, computational tools, and other components to create more sophisticated and useful applications.\n\nHere\'s a breakdown of what LangChain is and why it\'s so important:\n\n---\n\n### The Problem LangChain Solves\n\nLLMs like GPT-4 or Claude are incredibly powerful, but they have inherent limitations:\n\n1.  **Lack of Real-time Information:** Their knowledge is limited to their training data, which has a cutoff date. They can\'t access current events or proprietary company data directly.\n2.  **Inability to Perform Actions:** They can generate text, but they can

In [ ]:
thread_id = '1'
while True:
    
    user_query = input('Type here: ')
    print('User: ', user_query)
    
    if user_query.strip().lower() in ['end', 'break', 'bye', 'exit']:
        break
    
    config = {'configurable': {'thread_id': thread_id}}
    
    response = workflow.invoke({'messages': [HumanMessage(content= user_query)]}, config= config)
    print('AI: ', response['messages'][-1].content)

User:  hi my name is airi
AI:  Hi Airi, it's nice to meet you!
User:  what is my name?
AI:  Your name is Airi!
User:  who told u?
AI:  You did! You told me in your very first message when you said "hi my name is airi".
User:  exit
AI:  Okay, Airi.
User:  bye
